### Setup

In [1]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if PAGEINDEX_API_KEY:
    print("PageIndex API key loaded")
else:
    print("PageIndex API key not loaded")

if GROQ_API_KEY:
    print("Groq API key loaded")
else:
    print("Groq API key not loaded")

PageIndex API key loaded
Groq API key loaded


In [2]:
from pageindex import PageIndexClient

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

### Uploading Document

In [3]:
PDF_PATH = "./pdfs/Research_paper.pdf"

print(f"Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print("Uploaded")
print(f"Document ID: {doc_id}")

Uploading: ./pdfs/Research_paper.pdf
Uploaded
Document ID: pi-cmrlz33ke00ff01o40h7v8m79


In [4]:
print("Building Tree Index")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"Status: {status}")

    if status == "completed":
        print("Tree Index ready")
        break
    elif status == "failed":
        print("Process Failed")
        break

    time.sleep(5)

Building Tree Index
Status: completed
Tree Index ready


### Inspecting the Tree Structure

In [6]:
print("Before get_tree()")
tree_result = pi_client.get_tree(doc_id, node_summary=True)
print("After get_tree()")
pageindex_tree = tree_result.get("result",[])

print(f"Top level sections: {len(pageindex_tree)}")
print(f"\n Raw Tree: ")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

Before get_tree()
After get_tree()
Top level sections: 1

 Raw Tree: 
{
  "title": "VOCALIS: Bridging the Domain Gap for Robust Zero-Shot Voice Conversion from Synthetic",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "The paper introduces VOCALIS, a compact adapter designed to resolve the performance degradation in zero-shot voice conversion systems when conditioned on synthetic TTS audio. By aligning synthetic speaker embeddings with natural speech representations, the 330k-parameter adapter integrates into the YourTTS pipeline to improve cosine similarity by 75.5%. Evaluations across multiple languages and perceptual tests confirm that VOCALIS enhances speaker identity and naturalness, offering a robust, generalizable, and efficient solution for synthetic-to-natural voice conversion.",
  "text": "# VOCALIS: Bridging the Domain Gap for Robust Zero-Shot Voice Conversion from Synthetic\n\nTata Puruhutika Rao\nSchool of Computer Science and Engineering\nVIT-AP University A

In [7]:
# Pretty print full tree

def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview"""
    for node in nodes:
        prefix = " " * indent + ("|_ " if indent > 0 else "")
        page = node.get("page_index","?")
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent+1)
    
print("Full Document Structure:\n")
print_tree(pageindex_tree)

Full Document Structure:

[0000] VOCALIS: Bridging the Domain Gap for Robust Zero-Shot Voice Conversion from Synthetic (p.1)
 |_ [0001] I. INTRODUCTION (p.1)
 |_ [0002] II. RELATED WORK (p.1)
 |_ [0003] III. PROPOSED METHODOLOGY (p.2)
  |_ [0004] A. Problem Formulation (p.2)
  |_ [0005] B. VOCALIS Architecture (p.2)
  |_ [0006] C. Loss and Optimization (p.3)
  |_ [0007] A. Pairing and Splits (p.3)
  |_ [0008] B. Reproducibility (p.3)
  |_ [0009] C. Implementation Details (p.3)
 |_ [0010] V. EVALUATION METRICS (p.4)
  |_ [0011] A. Embedding-space metrics (p.4)
  |_ [0012] B. Perceptual evaluation (p.4)
  |_ [0013] C. Computational and reproducibility metrics (p.5)
  |_ [0014] D. Ablation studies (p.5)
 |_ [0015] VI. RESULTS AND DISCUSSIONS (p.6)
 |_ [0016] VIII. CONCLUSION (p.7)


In [8]:
# Count total nodes

def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"Total nodes in tree: {total}")

Total nodes in tree: 17


### LLM Tree Search

In [18]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

def llm_tree_search(query: str, tree: list, model: str = "openai/gpt-oss-120b") -> dict:
    """
    Core pageIndex Retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.

    Returns: dist with 'thinking' and 'node_list'
    """

    # compress tree to save tokens - send only title and short summaries
    def compress(nodes):
        out=[]
        for n in nodes:
            entry={
                "node_id":n["node_id"],
                "title": n["title"],
                "page": n.get("page_index","?"),
                "summary":n.get("text","")[:150]
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)

    prompt = f"""You are given a query and a document's tree structure. Your task is to identify which node ids most likely contain the answer to the query. Think step by step about which sections are relevant
        Query: {query}
        Document Tree: {json.dumps(compressed_tree, indent=2)}

        Reply only in this format:
        {{
            "thinking": "<your step by step reasoning>",
            "node_list": ["node_id1", "node_id2"]
        }}"""
    
    llm = ChatGroq(api_key=os.getenv("GROQ_API_KEY"), model="openai/gpt-oss-120b")
    response = llm.invoke([HumanMessage(content=prompt)])

    return json.loads(response.content)

In [19]:
# Test with sample query

query = "What is the limitations of VOcalis?"

print(f"Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("LLM Reasoning:")
print(result.get("thinking","N/A"))
print()
print("Selected Node IDS:", result.get("node_list", []))

Query: What is the limitations of VOcalis?

LLM Reasoning:
The query asks for the limitations of VOcalis. Limitations are typically discussed in sections that analyze results or summarize the work, such as a 'Results and Discussions' or 'Conclusion' section. In the provided tree, node 0015 corresponds to the 'VI. RESULTS AND DISCUSSIONS' section, and node 0016 corresponds to the 'VIII. CONCLUSION' section. These are the most likely places where the authors would mention any constraints or drawbacks of the VOcalis approach. No other sections (e.g., methodology or evaluation metrics) are expected to detail limitations directly.

Selected Node IDS: ['0015', '0016']


### Complete End-to-End RAG Pipeline

In [20]:
def find_nodes_by_ids(tree: list, target_ids:list) -> list:
    """Recursively walk the tree and collect nodes matching target ids"""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    
    return found

In [32]:
def generate_answer(query: str, nodes: list, modesl: str = "openai/gpt-oss-120b") -> str:
    """
    Takes retrieved nodes as  context and generates a grounded answer. Instructs the LLM to cite section titles and page numbers
    """

    if not nodes:
        return "No relevant sections found!"
    
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text','Content not available')}"
        )
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""
        You are an expert document analyst. Answer the question using only the provided context. For every claim you make, cite the section title and page number in the parenthesis. Be concise and precise.Reply ONLY with valid JSON format...
        Question: {query}
        Context: {context} 
        Answer:
    """

    llm = ChatGroq(api_key=os.getenv("GROQ_API_KEY"), model="openai/gpt-oss-120b")
    response = llm.invoke([HumanMessage(content=prompt)])

    return response.content

In [33]:
def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end to end PageIndex RAG Pipeline:

    Step 1: LLM Tree Search -> finds relevant node_ids
    Step 2: Node Retrieval -> fetches section content
    Step 3: Answer Generation -> produces cited answer 
    """

    if verbose:
        print(f"{'='*55}")
        print(f"Query: {query}")
        print(f"{'='*55}")

    # Step 1 - Tree Search
    search_result = llm_tree_search(query, tree)
    node_ids = search_result.get("node_list",[])

    if verbose:
        print(f"\n Reasoning: {search_result.get('thinking',' ')[:200]}...")
        print(f"Retrieved Nodes: {node_ids}")
    
    # Step 2 - Retrieve Nodes
    nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print(f"Sections found: {[n['title'] for n in nodes]}")

    # Step 3 - Generate Answer
    answer = generate_answer(query, nodes)

    if verbose:
        print(f"\nAnswer: \n{answer}")
    
    return answer

In [34]:
answer = vectorless_rag(
    query = "What are the limitations of Vocalis?",
    tree = pageindex_tree
)

Query: What are the limitations of Vocalis?

 Reasoning: The query asks for the limitations of Vocalis. Information about a method's shortcomings is typically discussed in the Results and Discussions section, where authors analyze performance gaps and chall...
Retrieved Nodes: ['0015', '0016']
Sections found: ['VI. RESULTS AND DISCUSSIONS', 'VIII. CONCLUSION']

Answer: 
{
  "limitations": [
    "The adapter does not correct prosodic or rhythmic artifacts inherited from the synthetic TTS reference, so such artifacts can bleed into the converted output, especially with uniform pitch movements or overly compressed timing (VIII. CONCLUSION, p.7).",
    "The method operates only at the embedding level and does not address sequence‑level cues or prosodic statistics, leaving a gap in handling expressive or stylistically varied synthetic inputs (VIII. CONCLUSION, p.7).",
    "Generalization to more expressive synthetic styles (e.g., emotional, whispered, or stylized voices) remains untested an

In [35]:
test_queries = [
    "What is the architecture of Vocalis?",
    "What are the results obtained in Vocalis?",
    "What are the limitations of Vocalis?"
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Question: {q}")
    print(f"Answer: {ans[:300]}")
    print("-" * 55)


Question: What is the architecture of Vocalis?
Answer: {
  "answer": "Vocalis is a three‑layer feed‑forward adapter network with a bottleneck structure, residual weighting, and explicit layer‑normalization. It takes a 256‑dimensional synthetic speaker embedding as input, applies a dropout‑ReLU linear layer (512×256), layer‑norm, a second dropout‑ReLU li
-------------------------------------------------------

Question: What are the results obtained in Vocalis?
Answer: {
  "speaker_encoder_SECS": {
    "HASP(256d)": {
      "value": 0.862,
      "source": "D. Ablation studies, p.5"
    },
    "x-vector(512d)": {
      "value": 0.788,
      "source": "D. Ablation studies, p.5"
    },
    "ECAPA-TDNN(192d)": {
      "value": 0.804,
      "source": "D. Ablation studi
-------------------------------------------------------

Question: What are the limitations of Vocalis?
Answer: {
  "limitations": [
    {
      "limitation": "The adapter does not correct prosodic or rhythmic artifacts inher